In [1]:
import sys
print(sys.executable)

c:\Users\brije\OneDrive\Documents\Learning\insurance-agency-ai-analyst\.venv\Scripts\python.exe


## Step 1: set up paths and read only the header

In [2]:
import csv
import sqlite3
from pathlib import Path

CSV_PATH = Path("../data/raw/finalapi.csv")
DB_PATH = Path("../database/insurance.db")

In [3]:
with CSV_PATH.open(newline="", encoding="utf-8") as f:
    header = next(csv.reader(f))

In [4]:
print(len(header))
print(header[:5])

49
['AGENCY_ID', 'PRIMARY_AGENCY_ID', 'PROD_ABBR', 'PROD_LINE', 'STATE_ABBR']


## Step 2: create the table and stream the rows in

In [5]:
DB_PATH.parent.mkdir(exist_ok=True)
conn = sqlite3.connect(DB_PATH)

In [6]:
columns_sql = ", ".join(f'"{col}" TEXT' for col in header)
conn.execute("DROP TABLE IF EXISTS stg_agency_performance")
conn.execute(f"CREATE TABLE stg_agency_performance ({columns_sql})")

In [7]:
placeholders = ", ".join("?" for _ in header)
with CSV_PATH.open(newline="", encoding="utf-8") as f:
    reader = csv.reader(f)
    next(reader)  # skip header row
    conn.executemany(
        f"INSERT INTO stg_agency_performance VALUES ({placeholders})", reader
    )
conn.commit()

## Step 3: check the load

In [8]:
print(conn.execute("SELECT COUNT(*) FROM stg_agency_performance").fetchone())
print(conn.execute("SELECT typeof(WRTN_PREM_AMT), WRTN_PREM_AMT FROM stg_agency_performance LIMIT 1").fetchone())

(213328,)
('text', '3074.63')


In [9]:
print(conn.execute("SELECT * FROM stg_agency_performance limit 10").fetchall())

[('3', '3', 'BOILERMACH', 'CL', 'IN', '2005', '0', '0', '0', '40', '3074.63', '149', '3005.77', '0', '8', '99999', '0', '99999', '99999', '1957', '14', '85', '48', 'N', 'Unknown', '99999', '99999', '99999', '99999', '99999', '99999', '99999', '99999', '0', '1', '0', '0', '0', '0', '0', '0', '0', '103', '50', '288', '0', '0', '0', '0'), ('3', '3', 'BOILERMACH', 'CL', 'IN', '2006', '0', '0', '0', '151', '6562.28', '5438.55', '6433.44', '0', '12', '99999', '0', '0.69538969', '99999', '1957', '14', '85', '48', 'N', 'Unknown', '99999', '99999', '99999', '99999', '99999', '99999', '99999', '99999', '0', '1', '0', '0', '0', '0', '0', '0', '0', '103', '50', '288', '0', '0', '0', '0'), ('3', '3', 'BOILERMACH', 'CL', 'IN', '2007', '0', '0', '0', '40', '6205.11', '6562.28', '6357.09', '0', '12', '99999', '0', '0.472728542', '99999', '1957', '14', '85', '48', 'N', 'Unknown', '99999', '99999', '99999', '99999', '99999', '99999', '99999', '99999', '0', '1', '0', '0', '0', '0', '0', '0', '0', '103', 

In [10]:
from html import escape
from IPython.display import HTML, display

def show(sql, limit=10):
    cur = conn.execute(f"SELECT * FROM ({sql}) LIMIT {limit}")
    cols = [d[0] for d in cur.description]
    rows = cur.fetchall()

    head = "".join(f"<th>{escape(c)}</th>" for c in cols)
    body = "".join(
        "<tr>" + "".join(f"<td>{escape(str(v))}</td>" for v in r) + "</tr>"
        for r in rows
    )
    display(HTML(f"<table><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table>"))

In [11]:
show("""
    SELECT AGENCY_ID, PROD_ABBR, PROD_LINE, STATE_ABBR, STAT_PROFILE_DATE_YEAR,
           WRTN_PREM_AMT, PRD_ERND_PREM_AMT, PRD_INCRD_LOSSES_AMT, LOSS_RATIO
    FROM stg_agency_performance
""")

AGENCY_ID,PROD_ABBR,PROD_LINE,STATE_ABBR,STAT_PROFILE_DATE_YEAR,WRTN_PREM_AMT,PRD_ERND_PREM_AMT,PRD_INCRD_LOSSES_AMT,LOSS_RATIO
3,BOILERMACH,CL,IN,2005,3074.63,3005.77,0,0
3,BOILERMACH,CL,IN,2006,6562.28,6433.44,0,0
3,BOILERMACH,CL,IN,2007,6205.11,6357.09,0,0
3,BOILERMACH,CL,IN,2008,5159.89,5315.01,0,0
3,BOILERMACH,CL,IN,2009,4527.71,4706.47,0,0
3,BOILERMACH,CL,IN,2010,4768.49,4653.02,8972.35,1.881591447
3,BOILERMACH,CL,IN,2011,4988.11,4790.67,3525.92,0.706864925
3,BOILERMACH,CL,IN,2012,4905.49,5005.71,0,0
3,BOILERMACH,CL,IN,2013,4545.51,4749.26,0,0
3,BOILERMACH,CL,IN,2014,3662.71,3838.8,0,0


## Step 4: Find 99999 values and blanks in every column

In [12]:
total = conn.execute("SELECT COUNT(*) FROM stg_agency_performance").fetchone()[0]

for col in header:
    n_99999, n_blank = conn.execute(f'''
        SELECT SUM(CAST("{col}" AS REAL) = 99999),
               SUM(TRIM("{col}") = '')
        FROM stg_agency_performance
    ''').fetchone()
    if n_99999 or n_blank:
        print(f"{col:28} 99999: {n_99999:>7} ({n_99999/total:6.1%})   blank: {n_blank}")

PRIMARY_AGENCY_ID            99999:   37874 ( 17.8%)   blank: 0
PREV_POLY_INFORCE_QTY        99999:   13893 (  6.5%)   blank: 0
NB_WRTN_PREM_AMT             99999:       1 (  0.0%)   blank: 0
PREV_WRTN_PREM_AMT           99999:   13893 (  6.5%)   blank: 0
RETENTION_RATIO              99999:  129978 ( 60.9%)   blank: 0
LOSS_RATIO                   99999:   51232 ( 24.0%)   blank: 0
LOSS_RATIO_3YR               99999:   49117 ( 23.0%)   blank: 0
GROWTH_RATE_3YR              99999:   91067 ( 42.7%)   blank: 0
AGENCY_APPOINTMENT_YEAR      99999:    5639 (  2.6%)   blank: 0
ACTIVE_PRODUCERS             99999:    5639 (  2.6%)   blank: 0
MAX_AGE                      99999:    5639 (  2.6%)   blank: 0
MIN_AGE                      99999:    5639 (  2.6%)   blank: 0
PL_START_YEAR                99999:   82200 ( 38.5%)   blank: 0
PL_END_YEAR                  99999:  209174 ( 98.1%)   blank: 0
COMMISIONS_START_YEAR        99999:  141937 ( 66.5%)   blank: 0
COMMISIONS_END_YEAR          99999:  205